# Bayesian vs frequentist inference at low statistics

With the full sample the posterior is Gaussian and, with flat priors, the Bayesian and frequentist results coincide (Bayesian_Nominal.ipynb vs the frequentist fit). This notebook looks for the regime where they differ: the same likelihood is analysed in both ways on pseudo-datasets of decreasing size.

- Model: six-parameter model (window [5150, 5450] MeV, Gaussian signal, one exponential background, shape and yield shared by the two charges), identifiable down to a few tens of events (`lowstat.py`).
- Truth: its fit to the data, with the yields rescaled to N = 50, 150, 500, 2000 candidates (Poisson-fluctuated).
- Bayesian: emcee, flat priors, median and equal-tailed 68%/95% credible intervals. Frequentist: MLE (iminuit) of the same likelihood, Hesse and Minos (profile likelihood, Wilks) 68%/95% intervals.
- Output: `results/lowstat/toys.jsonl` (checkpoint, resumable), `results/lowstat/summary.json`. Runtime ~1 h on 9 cores.


In [1]:
import os, sys, json
import numpy as np
sys.path.insert(0, ".")
import bayesian_nominal as bn
import lowstat as ls

DATA_DIR = "../../opendata-project/Data"
OUT = os.path.join(os.getcwd(), "results", "lowstat"); os.makedirs(OUT, exist_ok=True)
N_GRID = [50, 150, 500, 2000]
N_TOYS = 300
NSTEPS = 4000
N_WORKERS = max(1, (os.cpu_count() or 2) - 1)


## Truth: the six-parameter model fitted to the data

In [2]:
path = os.path.join(OUT, "truth.json")
if os.path.exists(path):
    truth = json.load(open(path)); print("[resume]", path)
else:
    d = bn.select(bn.load_candidates(DATA_DIR), ls.CFG)
    r = bn.fit(d.m.to_numpy(), d.q.to_numpy(), ls.CFG, nsteps=10000)
    truth = {"n": r["n"], "map": r["map"], "A": r["A"], "chain_over_tau_min": r["chain_over_tau_min"]}
    json.dump(truth, open(path, "w"), indent=1); print("[saved]", path)
print(truth["n"], {k: round(v, 5) for k, v in truth["map"].items()})


[saved] /Users/gmoretti/Work/wt_bayes/LHCb/Code_general/Bayesian/results/lowstat/truth.json
17839 {'N_s': 15283.69879, 'N_b': 2555.30122, 'A': -0.04963, 'mu': 5284.41191, 'sigma': 17.94415, 'lam1_all': 0.0031}


## Pseudo-datasets

In [3]:
tasks = [(f"{N}_{i}", N, truth["map"], 100000 * N + i, NSTEPS) for N in N_GRID for i in range(N_TOYS)]
toys = bn.run_parallel(tasks, ls.task, os.path.join(OUT, "toys.jsonl"), key=lambda r: r["key"], n_workers=N_WORKERS, log_every=50)


/Users/gmoretti/Work/wt_bayes/LHCb/Code_general/Bayesian/results/lowstat/toys.jsonl: 0 already done, 1200 to run


  50/1200 done, 7.0 min elapsed, ~160.4 min left


  100/1200 done, 13.2 min elapsed, ~145.6 min left


  150/1200 done, 19.4 min elapsed, ~136.1 min left


  200/1200 done, 25.9 min elapsed, ~129.6 min left


...

  1000/1200 done, 127.6 min elapsed, ~25.5 min left


  1050/1200 done, 132.1 min elapsed, ~18.9 min left


  1100/1200 done, 136.4 min elapsed, ~12.4 min left


  1150/1200 done, 141.0 min elapsed, ~6.1 min left


  1200/1200 done, 145.7 min elapsed, ~0.0 min left


## Coverage, bias and interval shape vs sample size

In [4]:
def frac(v):
    v = [x for x in v if x is not None]
    p = float(np.mean(v)); return p, float(np.sqrt(p * (1 - p) / len(v))), len(v)
summary = {}
for N in N_GRID:
    t = [x for x in toys if x["n_total"] == N]
    a0 = t[0]["A_true"]
    b_med = np.array([x["bayes"]["median"] for x in t]); f_hat = np.array([x["freq"]["A"] for x in t])
    b_sd = np.array([x["bayes"]["sd"] for x in t]); f_he = np.array([x["freq"]["hesse"] for x in t])
    s = {"n_toys": len(t), "A_true": a0, "mean_n": float(np.mean([x["n"] for x in t]))}
    for lab, key in (("bayes68", ("bayes", "in68")), ("bayes95", ("bayes", "in95")),
                     ("minos68", ("freq", "in68_minos")), ("minos95", ("freq", "in95_minos")),
                     ("hesse68", ("freq", "in68_hesse")), ("hesse95", ("freq", "in95_hesse"))):
        s[lab] = frac([x[key[0]][key[1]] for x in t])
    s["bias_bayes_median"] = [float(np.mean(b_med - a0)), float(np.std(b_med) / np.sqrt(len(t)))]
    s["bias_mle"] = [float(np.nanmean(f_hat - a0)), float(np.nanstd(f_hat) / np.sqrt(len(t)))]
    s["spread_bayes_median"] = float(np.std(b_med)); s["spread_mle"] = float(np.nanstd(f_hat))
    s["mean_posterior_sd"] = float(np.mean(b_sd)); s["mean_hesse"] = float(np.nanmean(f_he))
    s["mean_abs_median_minus_mle"] = float(np.nanmean(np.abs(b_med - f_hat)))
    s["mean_rel_ci68_asymmetry"] = float(np.mean([x["bayes"]["skew_68"] / (x["bayes"]["ci68"][1] - x["bayes"]["ci68"][0]) for x in t]))
    s["frac_freq_valid"] = float(np.mean([x["freq"]["valid"] for x in t]))
    s["min_chain_over_tau"] = float(np.min([x["bayes"]["chain_over_tau_min"] for x in t]))
    summary[N] = s
    print(f"N={N:5d}  cov68: Bayes {s['bayes68'][0]:.3f}+-{s['bayes68'][1]:.3f}  Minos {s['minos68'][0]:.3f}  Hesse {s['hesse68'][0]:.3f} | "
          f"cov95: Bayes {s['bayes95'][0]:.3f}  Minos {s['minos95'][0]:.3f}  Hesse {s['hesse95'][0]:.3f} | "
          f"bias med {s['bias_bayes_median'][0]:+.4f}  MLE {s['bias_mle'][0]:+.4f} | |med-MLE| {s['mean_abs_median_minus_mle']:.4f} | valid {s['frac_freq_valid']:.3f}")
json.dump({"N_grid": N_GRID, "summary": summary, "model": "lowstat (6 parameters, [5150,5450])"}, open(os.path.join(OUT, "summary.json"), "w"), indent=1)
print("[saved]", os.path.join(OUT, "summary.json"))


N=   50  cov68: Bayes 0.720+-0.026  Minos 0.703  Hesse 0.697 | cov95: Bayes 0.940  Minos 0.927  Hesse 0.923 | bias med -0.0070  MLE -0.0085 | |med-MLE| 0.0075 | valid 1.000
N=  150  cov68: Bayes 0.673+-0.027  Minos 0.670  Hesse 0.670 | cov95: Bayes 0.950  Minos 0.947  Hesse 0.947 | bias med -0.0044  MLE -0.0051 | |med-MLE| 0.0024 | valid 1.000
N=  500  cov68: Bayes 0.683+-0.027  Minos 0.687  Hesse 0.687 | cov95: Bayes 0.937  Minos 0.937  Hesse 0.937 | bias med -0.0021  MLE -0.0022 | |med-MLE| 0.0011 | valid 1.000
N= 2000  cov68: Bayes 0.717+-0.026  Minos 0.723  Hesse 0.723 | cov95: Bayes 0.983  Minos 0.987  Hesse 0.987 | bias med +0.0010  MLE +0.0009 | |med-MLE| 0.0006 | valid 1.000
[saved] /Users/gmoretti/Work/wt_bayes/LHCb/Code_general/Bayesian/results/lowstat/summary.json
